In [1]:
%load_ext autoreload
%autoreload 2

import os

os.chdir("../../")

### Split the human annotated data into train/test

- Run `python -m src.data.restructure_entrainment_coding_data --output_csv` first.

In [2]:
import pandas as pd

dogs_df = pd.read_excel("baskets_dogs_data/dogs.xlsx")
baskets_df = pd.read_excel("baskets_dogs_data/baskets.xlsx")
pairs = dogs_df.columns.tolist()[3:]
pairs

['Pair_20',
 'Pair_21',
 'Pair_22',
 'Pair_23',
 'Pair_24',
 'Pair_34',
 'Pair_35',
 'Pair_37',
 'Pair_43',
 'Pair_45']

In [3]:
from glob import glob

annotated_data = []
pairs_ixes = set(p.split("_")[-1] for p in pairs)
csv_files = glob("baskets_dogs_data/restructured_entrainment_coding_data/**/*.csv", recursive=True)

for csv_file in csv_files:
    parts = csv_file.split("/")
    object_category = parts[2]
    pair_ix = parts[3].replace("pair", "").replace(".csv", "")

    df = pd.read_csv(csv_file)
    df.insert(0, "object_category", object_category)
    df.insert(1, "pair_ix", pair_ix)

    split = "train" if pair_ix not in pairs_ixes else "test"
    df.insert(2, "split", split)
    annotated_data.append(df)


annotated_df = pd.concat(annotated_data).reset_index(drop=True)
annotated_df

,object_category,pair_ix,split,FirstRoundCardIndex,Round1,Round2,Round3,Round4
0,baskets,45,test,Card1,"rectangle shaped, handle in middle, no lid, empty","rectangle shaped with handle in middle, no lid",rectangle shaped with handle in middle,rectangular shaped with handle in middle
1,baskets,45,test,Card2,"Easter basket, pink things, pink wood, twisted...",Easter basket,Easter basket,Easter basket
2,baskets,45,test,Card3,"circular, criss-cross box, handle, one of them...",circular with criss-crosses and little black s...,"confusing one that's circular, with criss-cros...",confusing one with black stuff inside
3,baskets,45,test,Card4,"doesn't have a lid, wide, circle, really tiny ...","widest, bowl shaped, no handles and no lid","most circular and widest, with no lid and no h...","wide bowl, biggest, most circular"
4,baskets,45,test,Card5,"pink ribbon going around it, small, criss-cros...","smallest with pink ribbon going around it, no lid","smallest, pink ribbon going around it",smallest with pink ribbon going around
...,...,...,...,...,...,...,...,...
595,dogs,18,train,Card6,Doberman pinscher,Doberman pinscher,Doberman pinscher,Doberman pinscher
596,dogs,18,train,Card7,"facing left side, long ears, in stretching pos...","facing left position, long, droopy ear, beige ...","long droopy ears, yellowish beige","long, droopy ears, yellowish beige"
597,dogs,18,train,Card8,"resembles a Rotweiler, ears pointed up, as tho...","aggressive, attack position, black snozzle, ru...","attack position, ears protruded upward",attack position
598,dogs,18,train,Card9,"resembles a fox, resembles a wild dog, wild an...","resembles a wild animal, black and white",wild animal,wild animal


In [4]:
annotated_df[annotated_df.split == "test"].pair_ix.unique()

array(['45', '35', '22', '20', '23', '37', '21', '43', '34', '24'],
      dtype=object)

In [5]:
annotated_df.to_csv("baskets_dogs_data/entrainment_coding_annotated_data.csv", index=False)

In [6]:
train_df = annotated_df[annotated_df.split == "train"]
test_df = annotated_df[annotated_df.split == "test"]

### Few-shot exemplars

- 2-shots. One for dogs, the other for baskets.

In [13]:
import json

exemplar = train_df.sample(2)
exemplars = exemplar[["Round1", "Round2", "Round3", "Round4"]].T.to_dict().values()
exemplars = [json.dumps(exemplar, indent=4) for exemplar in exemplars]
for exemplar in exemplars:
    print(exemplar)

{
    "Round1": "doesn't have handle, tip of it is thicker than rest of body, brownish color, weaves are in squares if you look at it directly",
    "Round2": "half circle, no handles, top tip of it is a little bit thicker than rest of body",
    "Round3": "tip which is a little bit thicker than rest of body",
    "Round4": "tip that is a little bit larger than body, looks a little bit thicker"
}
{
    "Round1": "looks like German shepherd, all black but with beigish nose, on all fours",
    "Round2": "looking in same direction, all black, outlined in yellow, but mainly black",
    "Round3": "Saint Bernard, blackish, up on all fours",
    "Round4": "like a Doberman"
}


## Prompting

- This is essentially an extractive summarization task

In [14]:
from string import Template

prompt_temp = f"""
This is an extractive summarization task. You will be provided with \
four excerpts from four different rounds of a conversation between two \
participants trying to match the same object. One partcipant is responsible \
for describing the object, while the other participant is responsible for \
selecting the target object from a set of other objects. \
Your task is to extract the descriptive phrases used by the describer \
in each round that specifically refer to the target object. Extract these \
phrases verbatim from the excerpts and output them in a JSON fomat as follows:

{{
    "Round1": "descriptive phrases from Round 1",
    "Round2": "descriptive phrases from Round 2",
    "Round3": "descriptive phrases from Round 3",
    "Round4": "descriptive phrases from Round 4"
}}

Here are two examples to illustrate the format:

Exemplar 1:

{exemplars[0]}

Exemplar 2:

{exemplars[1]}

Now, please extract the descriptive phrases from the following conversation rounds. Do not \
output any additional text other than the JSON object.

$excerpts
""".strip()

prompt_temp = Template(prompt_temp)
print(prompt_temp.template)

This is an extractive summarization task. You will be provided with four excerpts from four different rounds of a conversation between two participants trying to match the same object. One partcipant is responsible for describing the object, while the other participant is responsible for selecting the target object from a set of other objects. Your task is to extract the descriptive phrases used by the describer in each round that specifically refer to the target object. Extract these phrases verbatim from the excerpts and output them in a JSON fomat as follows:

{
    "Round1": "descriptive phrases from Round 1",
    "Round2": "descriptive phrases from Round 2",
    "Round3": "descriptive phrases from Round 3",
    "Round4": "descriptive phrases from Round 4"
}

Here are two examples to illustrate the format:

Exemplar 1:

{
    "Round1": "doesn't have handle, tip of it is thicker than rest of body, brownish color, weaves are in squares if you look at it directly",
    "Round2": "half

In [15]:
dogs_df["object_category"] = "dogs"
baskets_df["object_category"] = "baskets"
combined_df = pd.concat([dogs_df, baskets_df]).reset_index(drop=True)
combined_df.shape, combined_df.columns

((80, 14),
 Index(['Round', 'Order', 'TargetObjectIndex', 'Pair_20', 'Pair_21', 'Pair_22',
        'Pair_23', 'Pair_24', 'Pair_34', 'Pair_35', 'Pair_37', 'Pair_43',
        'Pair_45', 'object_category'],
       dtype='object'))

In [16]:
def make_excerpts_text(excerpts):
    return "\n\n".join([f"### Excerpt from Round {i+1}\n\n{text}" 
                        for i, text in enumerate(excerpts)])


pairs = ['Pair_20', 'Pair_21', 'Pair_22', 'Pair_23', 'Pair_24', 
         'Pair_34', 'Pair_35', 'Pair_37', 'Pair_43', 'Pair_45',]
sub = combined_df[(combined_df['object_category'] == 'dogs')]
FirstRoundCardIndex = sub[sub.Round == 1].TargetObjectIndex
subsub = sub[(sub.TargetObjectIndex == FirstRoundCardIndex[0])]["Pair_20"]

excerpts = make_excerpts_text(subsub.values)
prompt = prompt_temp.substitute(excerpts=excerpts)
print(prompt)

This is an extractive summarization task. You will be provided with four excerpts from four different rounds of a conversation between two participants trying to match the same object. One partcipant is responsible for describing the object, while the other participant is responsible for selecting the target object from a set of other objects. Your task is to extract the descriptive phrases used by the describer in each round that specifically refer to the target object. Extract these phrases verbatim from the excerpts and output them in a JSON fomat as follows:

{
    "Round1": "descriptive phrases from Round 1",
    "Round2": "descriptive phrases from Round 2",
    "Round3": "descriptive phrases from Round 3",
    "Round4": "descriptive phrases from Round 4"
}

Here are two examples to illustrate the format:

Exemplar 1:

{
    "Round1": "doesn't have handle, tip of it is thicker than rest of body, brownish color, weaves are in squares if you look at it directly",
    "Round2": "half

### Test

In [17]:
from src.llms.openai import get_response


_, response = get_response(
    prompt,
    model_name="gpt-4o-mini"
)
print(response)

{
    "Round1": "dog standing up very straight, black and brown, his legs are brown, tail's got a little black on it, black all through his torso, basset hound",
    "Round2": "large basset hound, black and the brown legs, he's got his tail down, little black in it",
    "Round3": "the basset hound, big",
    "Round4": "the big basset hound"
}


### Deployment

In [18]:
test_df.columns

Index(['object_category', 'pair_ix', 'split', 'FirstRoundCardIndex', 'Round1',
       'Round2', 'Round3', 'Round4'],
      dtype='object')

In [19]:
from tqdm import tqdm

llm_annotations = []
p_bar = tqdm(total=2 * 10 * 10)
cols = ["object_category", "pair_ix", "FirstRoundCardIndex", "Model_Response"]

for object_category in ["dogs", "baskets"]:
    combined_df_sub = combined_df[combined_df.object_category == object_category]
    TargetObjectIndexes = combined_df_sub[combined_df_sub.Round == 1].TargetObjectIndex.unique()

    for pair in pairs:
        pair_ix = pair.replace("Pair_", "")

        for i, TargetObjectIndex in enumerate(TargetObjectIndexes):
            subsub = combined_df_sub[combined_df_sub.TargetObjectIndex == TargetObjectIndex][pair]

            excerpts = make_excerpts_text(subsub.values)
            prompt = prompt_temp.substitute(excerpts=excerpts)

            _, response = get_response(
                prompt,
                model_name="gpt-4o-mini"
            )

            llm_annotations.append({
                "object_category": object_category,
                "pair_ix": pair_ix,
                "FirstRoundCardIndex": f"Card{i+1}",
                "Model_Response": response
            })
            p_bar.update(1)
 
p_bar.close()
annotated_llm_df = pd.DataFrame(llm_annotations)[cols]
annotated_llm_df

  0%|          | 0/200 [00:00<?, ?it/s]

100%|██████████| 200/200 [07:25<00:00,  2.23s/it]


,object_category,pair_ix,FirstRoundCardIndex,Model_Response
0,dogs,20,Card1,"```json\n{\n ""Round1"": ""dog standing up ver..."
1,dogs,20,Card2,"{\n ""Round1"": ""standing, uh, with his two f..."
2,dogs,20,Card3,"{\n ""Round1"": ""seated position, sitting, he..."
3,dogs,20,Card4,"{\n ""Round1"": ""stout little, uh, black dog,..."
4,dogs,20,Card5,"{\n ""Round1"": ""the poodle"",\n ""Round2"": ..."
...,...,...,...,...
195,baskets,45,Card6,"{\n ""Round1"": ""light goldish basket, has li..."
196,baskets,45,Card7,"{\n ""Round1"": ""has a lid and um it's like g..."
197,baskets,45,Card8,"{\n ""Round1"": ""triangle…shaped basket"",\n ..."
198,baskets,45,Card9,"```json\n{\n ""Round1"": ""circular one, has a..."


In [20]:
annotated_llm_df.to_csv("baskets_dogs_data/entrainment_coding_llm_annotations.csv", index=False)

In [27]:
import re

def parse_llm_response(response):
    response = re.search(r"\{.*\}", response, re.DOTALL)
    if response:
        response = response.group(0)
    else:
        return {}
    
    try:
        parsed = json.loads(response)
    except json.JSONDecodeError:
        parsed = {}

    return parsed


for ix, row in annotated_llm_df.iterrows():
    parsed_response = parse_llm_response(row.Model_Response)
    
    for round_num in range(1, 5):
        annotated_llm_df.at[ix, f"Round{round_num}"] = parsed_response.get(f"Round{round_num}", None)

In [28]:
len(annotated_llm_df.dropna())

200

In [29]:
annotated_llm_df.head()

,object_category,pair_ix,FirstRoundCardIndex,Model_Response,Round1,Round2,Round3,Round4
0,dogs,20,Card1,"```json\n{\n ""Round1"": ""dog standing up ver...","dog standing up very straight, black and brown...","large basset hound, black and the brown legs, ...","the basset hound, big guy",the big basset hound
1,dogs,20,Card2,"{\n ""Round1"": ""standing, uh, with his two f...","standing, uh, with his two front legs, uh...st...",dog that looks like he's looking down his leg,"guy looking down his leg, The *schnauzer*","The schnauzer, looking down his leg"
2,dogs,20,Card3,"{\n ""Round1"": ""seated position, sitting, he...","seated position, sitting, head turned",brown dog that's sitting,sitting dog,"dog that's sitting, turned to the back"
3,dogs,20,Card4,"{\n ""Round1"": ""stout little, uh, black dog,...","stout little, uh, black dog, very short, littl...",weiner dog,weiner dog,weiner
4,dogs,20,Card5,"{\n ""Round1"": ""the poodle"",\n ""Round2"": ...",the poodle,the black poodle,the poodle,The poodle


In [30]:
annotated_llm_df.to_csv("baskets_dogs_data/entrainment_coding_llm_annotations.csv", index=False)

In [7]:
annotated_llm_df = pd.read_csv("baskets_dogs_data/entrainment_coding_llm_annotations.csv")

### Evaluation

In [9]:
from rouge import Rouge
from sentence_transformers import SentenceTransformer, util

def rouge_f1(system_summary: str, reference_summary: str):
    rouge = Rouge()
    scores = rouge.get_scores(system_summary, reference_summary)[0]

    r1_f1 = scores["rouge-1"]["f"]
    r2_f1 = scores["rouge-2"]["f"]
    rL_f1 = scores["rouge-l"]["f"]

    return r1_f1, r2_f1, rL_f1


# Load once (global) so you don't re-load the model on every call
_sbert_model = SentenceTransformer("all-MiniLM-L6-v2")

def sbert_score(text1: str, text2: str) -> float:
    """
    Compute SBERT cosine similarity between two strings.
    Returns a single float (typically in [0, 1] for normal sentences).
    """
    emb1 = _sbert_model.encode(text1, convert_to_tensor=True)
    emb2 = _sbert_model.encode(text2, convert_to_tensor=True)

    # util.cos_sim returns a 1x1 tensor here
    score = util.cos_sim(emb1, emb2).item()
    return float(score)

In [10]:
test_df.head(1)

,object_category,pair_ix,split,FirstRoundCardIndex,Round1,Round2,Round3,Round4
0,baskets,45,test,Card1,"rectangle shaped, handle in middle, no lid, empty","rectangle shaped with handle in middle, no lid",rectangle shaped with handle in middle,rectangular shaped with handle in middle


In [15]:
test_df["pair_ix"] = test_df["pair_ix"].astype(int)

/tmp/ipykernel_1385954/413259986.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df["pair_ix"] = test_df["pair_ix"].astype(int)


In [16]:
annotated_llm_df["pair_ix"] = annotated_llm_df["pair_ix"].astype(int)

In [17]:
annotated_llm_df.head(1)

,object_category,pair_ix,FirstRoundCardIndex,Model_Response,Round1,Round2,Round3,Round4
0,dogs,20,Card1,"```json\n{\n ""Round1"": ""dog standing up ver...","dog standing up very straight, black and brown...","large basset hound, black and the brown legs, ...","the basset hound, big guy",the big basset hound


In [18]:
evaluation_results = []

for object_category in test_df.object_category.unique():
    test_df_sub = test_df[test_df.object_category == object_category]
    annotated_llm_df_sub = annotated_llm_df[annotated_llm_df.object_category == object_category]

    for pair_ix in test_df_sub.pair_ix.unique():
        test_df_subsub = test_df_sub[test_df_sub.pair_ix == pair_ix]
        annotated_llm_df_subsub = annotated_llm_df_sub[annotated_llm_df_sub.pair_ix == pair_ix]

        for ix, row in test_df_subsub.iterrows():
            FirstRoundCardIndex = row.FirstRoundCardIndex
            annotated_row = annotated_llm_df_subsub[
                annotated_llm_df_subsub.FirstRoundCardIndex == FirstRoundCardIndex
            ].iloc[0]

            for round_num in range(1, 5):
                reference_summary = row[f"Round{round_num}"]
                system_summary = annotated_row[f"Round{round_num}"]

                r1_f1, r2_f1, rL_f1 = rouge_f1(system_summary, reference_summary)

                evaluation_results.append({
                    "object_category": object_category,
                    "pair_ix": pair_ix,
                    "FirstRoundCardIndex": FirstRoundCardIndex,
                    "round_num": round_num,
                    "rouge-1_f1": r1_f1,
                    "rouge-2_f1": r2_f1,
                    "rouge-L_f1": rL_f1,
                    "sbert_score": sbert_score(system_summary, reference_summary),
                })

evaluation_df = pd.DataFrame(evaluation_results)
evaluation_df

,object_category,pair_ix,FirstRoundCardIndex,round_num,rouge-1_f1,rouge-2_f1,rouge-L_f1,sbert_score
0,baskets,45,Card1,1,0.400000,0.142857,0.400000,0.690133
1,baskets,45,Card1,2,0.625000,0.266667,0.625000,0.785143
2,baskets,45,Card1,3,0.857143,0.307692,0.857143,0.944746
3,baskets,45,Card1,4,0.923077,0.500000,0.923077,0.981395
4,baskets,45,Card2,1,0.666667,0.360000,0.666667,0.878465
...,...,...,...,...,...,...,...,...
795,dogs,24,Card9,4,0.800000,0.750000,0.800000,0.908934
796,dogs,24,Card10,1,0.842105,0.631579,0.842105,0.891194
797,dogs,24,Card10,2,0.761905,0.604651,0.761905,0.906219
798,dogs,24,Card10,3,0.933333,0.714286,0.933333,0.976210


In [20]:
evaluation_df.groupby("round_num")[["rouge-1_f1", "rouge-2_f1", "rouge-L_f1", "sbert_score"]].describe().T

round_num                   1           2           3           4
rouge-1_f1  count  200.000000  200.000000  200.000000  200.000000
            mean     0.620296    0.656635    0.652695    0.693043
            std      0.206122    0.226837    0.222252    0.220746
            min      0.000000    0.000000    0.000000    0.000000
            25%      0.500000    0.542424    0.500000    0.553030
            50%      0.635255    0.700000    0.666667    0.727273
            75%      0.770105    0.818182    0.823529    0.845865
            max      1.000000    1.000000    1.000000    1.000000
rouge-2_f1  count  200.000000  200.000000  200.000000  200.000000
            mean     0.382353    0.414858    0.400994    0.417823
            std      0.234339    0.274800    0.303107    0.311215
            min      0.000000    0.000000    0.000000    0.000000
            25%      0.209619    0.222222    0.153846    0.142857
            50%      0.356471    0.418860    0.400000    0.428571
            75%      0.533333    0.600000    0.600000    0.626645
            max      1.000000    1.000000    1.000000    1.000000
rouge-L_f1  count  200.000000  200.000000  200.000000  200.000000
            mean     0.606738    0.653430    0.648922    0.689177
            std      0.211315    0.230507    0.225017    0.225425
            min      0.000000    0.000000    0.000000    0.000000
            25%      0.481117    0.533333    0.500000    0.553030
            50%      0.621820    0.700000    0.666667    0.727273
            75%      0.760142    0.818182    0.819519    0.845865
            max      1.000000    1.000000    1.000000    1.000000
sbert_score count  200.000000  200.000000  200.000000  200.000000
            mean     0.823202    0.824418    0.853600    0.864320
            std      0.121203    0.136497    0.112020    0.122698
            min      0.334573    0.450227    0.461105    0.429996
            25%      0.749782    0.723836    0.772793    0.797745
            50%      0.842312    0.844328    0.868152    0.890433
            75%      0.920822    0.943523    0.953529    0.971339
            max      1.000000    1.000000    1.000000    1.000000

In [44]:
evaluation_df.to_csv("notebooks/outputs/llm_entrainment_coding_evaluation.csv", index=False)

In [21]:
combined_df = []
cols = ["object_category", "pair_ix", "FirstRoundCardIndex", "round_ix", "human_extracted", "llm_extracted"]

for object_category in test_df.object_category.unique():
    test_df_sub = test_df[test_df.object_category == object_category]
    annotated_llm_df_sub = annotated_llm_df[annotated_llm_df.object_category == object_category]

    for pair_ix in test_df_sub.pair_ix.unique():
        test_df_subsub = test_df_sub[test_df_sub.pair_ix == pair_ix]
        annotated_llm_df_subsub = annotated_llm_df_sub[annotated_llm_df_sub.pair_ix == pair_ix]

        for ix, row in test_df_subsub.iterrows():
            FirstRoundCardIndex = row.FirstRoundCardIndex
            annotated_row = annotated_llm_df_subsub[
                annotated_llm_df_subsub.FirstRoundCardIndex == FirstRoundCardIndex
            ].iloc[0]

            for round_ix in range(1, 5):
                human_extracted = row[f"Round{round_ix}"]
                llm_extracted = annotated_row[f"Round{round_ix}"]

                combined_df.append({
                    "object_category": object_category,
                    "pair_ix": pair_ix,
                    "FirstRoundCardIndex": FirstRoundCardIndex,
                    "round_ix": round_ix,
                    "human_extracted": human_extracted,
                    "llm_extracted": llm_extracted,
                })

combined_df = pd.DataFrame(combined_df)
combined_df

,object_category,pair_ix,FirstRoundCardIndex,round_ix,human_extracted,llm_extracted
0,baskets,45,Card1,1,"rectangle shaped, handle in middle, no lid, empty","rectangle shaped basket, the handle in the middle"
1,baskets,45,Card1,2,"rectangle shaped with handle in middle, no lid",rectangle shaped one with the handle in the mi...
2,baskets,45,Card1,3,rectangle shaped with handle in middle,rectangle shaped one with the handle in the mi...
3,baskets,45,Card1,4,rectangular shaped with handle in middle,rectangular shaped with the handle in the middle
4,baskets,45,Card2,1,"Easter basket, pink things, pink wood, twisted...","like a Easter basket, has like pink…things, pi..."
...,...,...,...,...,...,...
795,dogs,24,Card9,4,black and white husky,the black and white husky dog
796,dogs,24,Card10,1,hair drapes all the way from his body to groun...,"his hair is--uh, it drapes all the way from hi..."
797,dogs,24,Card10,2,"terrier, all black, longest haired dog, short ...","all black, longest haired dog, short little th..."
798,dogs,24,Card10,3,short little stumpy dog with moppy hair,the short little stumpy dog with the moppy hair


In [22]:
from src.data.utils import pretty_print

pretty_print(combined_df.sample(5))

,object_category,pair_ix,FirstRoundCardIndex,round_ix,human_extracted,llm_extracted
540,dogs,20,Card6,1,Doberman pinscher,Doberman pinscher
401,dogs,45,Card1,2,"black stomach and back and legs, goldish legs, tail is long, blackish tip on tail","dog with the black stomach and back and legs, goldish legs, tail is long, blackish tip on the tail"
710,dogs,43,Card8,3,"standing tall, black mouth, with black around surrounding its eyes, ears stickin' up, chain collar","standing tall, black mouth with black around surrounding its eyes, ears stickin' up, kinda like a chain collar"
12,baskets,45,Card4,1,"doesn't have a lid, wide, circle, really tiny like the sewing things, widest one, biggest one, no handles, just like a bowl","doesn't have a lid, it's like wide, it's a circle, really tiny like the sewing things, the widest one, looks like it will be the biggest one"
630,dogs,37,Card8,3,"brown, black snout, ears stickin' up, short tail","brown one with the black snout and the ears stickin' up, short tail"


In [23]:
combined_df.to_csv("notebooks/outputs/llm_and_human_extractions_comparison.csv", index=False)